# S05: exact tools and durable reviewed actions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/04_tools_and_state.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

70-minute lab. Read-only model-selected tools and human-authorised writes have different authority. A natural-language instruction is not an approval.

Read [the worked business cases](https://github.com/cybertide-solutions/ai-platform-architect-labs/blob/main/handouts/05-worked-business-cases.md) first. Predict the Nova Q1 total (INR 65,000, two orders) and explain the reviewed INR 12,000 workstation request.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–15 min | Let SQL own financial facts.** Compare AsterWorks and Beacon, then attempt SQL injection through a supplier field. This allowlisted tool rejects unsupported arguments before parameterised SQL. There is no model-generated SQL execution.

In [ ]:
from engine.purchasing import dispatch,bounded_agent
from engine.approvals import ApprovalLedger
p=Purchases();args={'quarter':'2026-Q1','supplier':'Nova'}
print(dispatch('spend_summary',args,FINANCE,p))
print(dispatch('spend_summary',args,BEACON,p))
print(dispatch('spend_summary',{'quarter':'2026-Q1','supplier':"Nova' OR 1=1"},FINANCE,p))
print(dispatch('make_payment',args,FINANCE,p))

In [ ]:
from engine.purchasing import format_inr, render_spend_summary
facts=p.summary(FINANCE,args)
assert facts["total_minor"] == 6500000
assert facts["total_display"] == "INR 65,000.00"
print("Stored amount:", facts["total_minor"], facts["minor_unit"])
print("Displayed answer:", render_spend_summary(facts))
print("Business meaning: recorded purchase-order amounts, not proof of payments.")


**15–30 min | Fixed workflow or agent?** Run a native-tool loop only if the selected model supports tools. It is capped at three model steps and three calls per step. The displayed financial answer is rendered by code from successful tool results. The model's final financial prose is discarded. 6,500,000 paise must display as INR 65,000.00. A tool-free guess is not a supported result. For one known query, the fixed parse/validate/SQL workflow may be cheaper and easier to test. MCP standardises interaction; it does not grant authority or make a tool trustworthy.

In [ ]:
if LIVE:
    result=bounded_agent(client,'How much did we spend with Nova in 2026-Q1?',FINANCE,p)
    print(json.dumps(result,indent=2))
else:
    print('Native model-selected tool loop pending. The preceding SQL/tool checks executed locally.')

**30–55 min | Review, mutate, retry, restart.** A finance reviewer approves one exact draft for one principal with a five-minute expiry. The ledger binds tenant, principal, entitlement version and draft. Commit persists a local purchase request, not a payment. Retry after reopening the database must return the same receipt. Changed content must conflict.

In [ ]:
draft={'supplier':'Nova','amount_minor':1200000,'purpose':'Approved design workstation'}
with tempfile.TemporaryDirectory() as td:
    path=Path(td)/'actions.db'; ledger=ApprovalLedger(path)
    token=ledger.approve(FINANCE,BUYER,draft,now=1000)
    tampered=dict(draft,amount_minor=9900000)
    assert ledger.commit(BUYER,tampered,token,'request-1',now=1001)['status']=='review_required'
    first=ledger.commit(BUYER,draft,token,'request-1',now=1002)
    ledger.close();ledger=ApprovalLedger(path)
    replay=ledger.commit(BUYER,draft,token,'request-1',now=1003)
    conflict=ledger.commit(BUYER,tampered,token,'request-1',now=1004)
    print(first,replay,conflict)
    assert replay['receipt']==first['receipt'] and conflict['status']=='conflict'
    expired=ledger.approve(FINANCE,BUYER,draft,now=1000)
    assert ledger.commit(BUYER,draft,expired,'request-2',now=1400)['status']=='review_required'
    ledger.close()
p.close()

**55–70 min | Extend and defend.** Add a revoked-entitlement test and a wrong-tenant reviewer test. Draw proposal → validation → review → durable commit → receipt. Explain why the external purchasing system needs an idempotency contract/outbox and reconciliation; a local transaction cannot guarantee exactly-once delivery to an external system. Identity objects are trusted lab inputs, not authentication. List the IdP and audit controls needed for production.